In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# Keep the 2024-onwards data as the working dataset
df_new = pd.read_csv("data/lahore_aqi_historical_2024_onwards.csv")


In [ ]:
df_new

In [ ]:
# Remove rows with missing values in the working dataframe
df_new = df_new.dropna(inplace=True)


In [ ]:
df_new.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")


In [ ]:
# Impute missing values in the target column if any remain
df_new[["ground_pm25"]] = imputer.fit_transform(df_new[["ground_pm25"]])
df = df_new.copy()
data = df.copy()


In [ ]:
df_new.isnull().sum()

In [ ]:
data.isnull().sum()

In [ ]:
data.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
import matplotlib.pyplot as plt

df_new.boxplot(figsize=(12,6))
plt.show()

In [ ]:
# Use numeric_only=True to ignore the timestamp text column
Q1 = df.quantile(0.25, numeric_only=True)
Q3 = df.quantile(0.75, numeric_only=True)
IQR = Q3 - Q1

print(IQR)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

feature_cols = [
    col for col in df_new.columns
    if col not in ['timestamp', 'ground_pm25'] and pd.api.types.is_numeric_dtype(df_new[col])
]

# Ensure all feature columns are numeric before scaling
for col in feature_cols:
    df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

# Fit scaler on the same feature columns used for training
# and keep the target column unchanged

df_scale = df_new.copy()
df_scale[feature_cols] = scaler.fit_transform(df_new[feature_cols])


In [ ]:
X = df_scale[feature_cols]
y = df_scale["ground_pm25"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
df.head(5)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)


In [ ]:
rf_model.fit(X_train, y_train)


In [ ]:
y_pred = rf_model.predict(X_test)


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
# --- 3. XGBOOST REGRESSOR ---
from xgboost import XGBRegressor
import time

start_time = time.time()
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start_time
print(f"XGBoost trained in {xgb_time:.2f} seconds")

In [ ]:
# --- 4. LIGHTGBM REGRESSOR ---
# Usually faster and slightly more accurate for tabular time series
import lightgbm as lgb

start_time = time.time()
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)
lgb_time = time.time() - start_time
print(f"LightGBM trained in {lgb_time:.2f} seconds")

In [ ]:
# --- 5. EVALUATION METRICS ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    # MAPE calculation (safeguard against division by zero)
    mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1e-6))) * 100
    r2 = r2_score(y_test, y_pred)
    
    print(f"==== {name} Results ====")
    print(f"MAE  : {mae:.2f} µg/m³ (Average error)")
    print(f"RMSE : {rmse:.2f} µg/m³ (Penalizes large spikes)")
    print(f"MAPE : {mape:.2f}% (Percentage error)")
    print(f"R²   : {r2:.4f}")
    print()

evaluate_model("XGBoost", xgb_model, X_test, y_test)
evaluate_model("LightGBM", lgb_model, X_test, y_test)

In [ ]:
# --- 6. FEATURE IMPORTANCE (LightGBM) ---
import matplotlib.pyplot as plt
import seaborn as sns

importance = lgb_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis')
plt.title('Top  Most Important Features (LightGBM)')
plt.tight_layout()
plt.show()



In [ ]:
# --- 7. 72-HOUR FORECASTING ---
# Create a simple future feature table for the next 72 hours.
# In a real deployment, these values should come from weather forecasts and modeled AQI estimates.

future_hours = 72
last_timestamp = pd.to_datetime(df_new['timestamp'].max())

future_index = pd.date_range(start=last_timestamp + pd.Timedelta(hours=1), periods=future_hours, freq='h')

future_df = pd.DataFrame({
    'timestamp': future_index,
    'temperature': [df_new['temperature'].mean()] * future_hours,
    'humidity': [df_new['humidity'].mean()] * future_hours,
    'wind_speed': [df_new['wind_speed'].mean()] * future_hours,
    'pressure': [df_new['pressure'].mean()] * future_hours,
    'modeled_pm25': [df_new['modeled_pm25'].mean()] * future_hours,
    'modeled_pm10': [df_new['modeled_pm10'].mean()] * future_hours,
})

future_df['hour'] = future_df['timestamp'].dt.hour
future_df['day_of_week'] = future_df['timestamp'].dt.dayofweek
future_df['month'] = future_df['timestamp'].dt.month

# Keep the same column order as training data
future_features = future_df[feature_cols].copy()

# Make sure future features have the same dtypes as the training data
for col in feature_cols:
    future_features[col] = pd.to_numeric(future_features[col], errors='coerce')

# Scale future features with the same scaler used for training
future_scaled = scaler.transform(future_features)

# Predict next 72 hours of PM2.5 values
future_predictions = rf_model.predict(future_scaled)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'timestamp': future_df['timestamp'],
    'predicted_ground_pm25': future_predictions
})

print('--- 72-Hour AQI Forecast Preview ---')
print(forecast_df.head(10))



In [3]:
import pandas as pd
df=pd.read_csv("data/lahore_aqi_historical_2024_onwards.csv")
df.isnull().sum()

timestamp             0
temperature           0
humidity              0
wind_speed            0
pressure              0
rainfall              0
modeled_pm25          0
modeled_pm10          0
ozone                 0
no2                   0
no                    0
so2                   0
co                    0
aqi                   0
ground_pm25        7916
hour                  0
day                   0
day_of_week           0
week_of_year          0
month                 0
aqi_change_rate       0
aqi_lag_1             0
dtype: int64

In [4]:
df.shape

(22728, 22)